# Phase 2 Notebook 03G: Signal Diversity Engine

Purpose: add a diversity-aware signal selection layer after Notebook 03F and before Alpha Construction. This notebook identifies high-quality signals that are less redundant with each other, so later alpha construction can avoid being dominated by highly correlated variants.

Scope boundaries:
- This notebook does not create new signals.
- It does not change signal formulas, scoring, WFV, decay, regime, health, reproducibility, alpha construction, stress, freeze, portfolio, or ML logic.
- It writes only `signal_diversity_*` diagnostic and selection tables.


## 1. Imports and Config

In [1]:
from pathlib import Path
import sqlite3
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "2-Phase 2_Signal Expansion":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.db import get_db_path
from src.run_config import make_run_id, make_run_timestamp
from src.signal_diversity import (
    build_cluster_diversity_report,
    build_diversity_candidate_table,
    build_family_diversity_report,
    build_signal_panels,
    build_signal_similarity_matrix,
    compute_diversity_diagnostics,
    greedy_diversity_selection,
    load_candidate_signal_rows,
    load_signal_diversity_inputs,
)
from src.signal_diversity_storage import SIGNAL_DIVERSITY_TABLES, save_signal_diversity_outputs

DB_PATH = get_db_path()
DIVERSITY_VERSION = "phase2_signal_diversity_v1"
CORRELATION_THRESHOLD = 0.85
MIN_SELECTED = 3
INCLUDE_WATCHLIST = False
INCLUDE_ORTHOGONAL_DIVERSIFIERS = True
ORTHOGONAL_DIVERSIFIER_VERSION = "phase2_orthogonal_signals_v2"
ORTHOGONAL_DIVERSIFIER_MIN_HEALTH_SCORE = 60
ORTHOGONAL_DIVERSIFIER_MIN_PASS_RATE = 0.60

pd.set_option("display.max_columns", 200)
DB_PATH

PosixPath('/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db')

## 2. Create run_id / timestamp

In [2]:
run_id = make_run_id(prefix="phase2_signal_diversity")
run_timestamp = make_run_timestamp()

run_id, run_timestamp

('phase2_signal_diversity_20260510_225559', '2026-05-10 22:55:59')

## 3. Load health and reproducibility inputs

In [3]:
inputs = load_signal_diversity_inputs(db_path=DB_PATH)

input_shapes = pd.DataFrame(
    [
        {"input_name": name, "n_rows": len(df), "n_columns": len(df.columns)}
        for name, df in inputs.items()
    ]
)
display(input_shapes)

with sqlite3.connect(DB_PATH) as conn:
    candidate_best_horizon_gap = pd.read_sql_query(
        """
        SELECT
            c.signal_name,
            COALESCE(q.status, 'MISSING_QUALITY_GATE') AS quality_gate_status,
            q.missing_pct,
            q.finite_pct,
            COUNT(*) AS candidate_rows,
            COUNT(DISTINCT c.Date) AS n_dates,
            COUNT(b.signal_name) AS best_horizon_rows
        FROM candidate_signals_current c
        LEFT JOIN signal_best_horizon_current b
            ON b.signal_name = c.signal_name
        LEFT JOIN candidate_signal_quality_gate_current q
            ON q.signal_name = c.signal_name
        GROUP BY c.signal_name, q.status, q.missing_pct, q.finite_pct
        HAVING best_horizon_rows = 0
        ORDER BY quality_gate_status, c.signal_name
        """,
        conn,
    )

if not candidate_best_horizon_gap.empty:
    print(
        "Signal exists in candidate_signals_current but has no best-horizon scoring row, "
        "so it is excluded from diversity analysis."
    )
    display(candidate_best_horizon_gap)

,input_name,n_rows,n_columns
0,health,92,31
1,reproducibility_gate,1,19


## 4. Build diversity candidate universe

In [4]:
diversity_candidates = build_diversity_candidate_table(
    health=inputs["health"],
    reproducibility_gate=inputs["reproducibility_gate"],
    include_watchlist=INCLUDE_WATCHLIST,
    include_orthogonal_diversifiers=INCLUDE_ORTHOGONAL_DIVERSIFIERS,
    orthogonal_diversifier_version=ORTHOGONAL_DIVERSIFIER_VERSION,
    orthogonal_diversifier_min_health_score=ORTHOGONAL_DIVERSIFIER_MIN_HEALTH_SCORE,
    orthogonal_diversifier_min_pass_rate=ORTHOGONAL_DIVERSIFIER_MIN_PASS_RATE,
)

eligible_candidates = diversity_candidates.loc[diversity_candidates["eligible_for_diversity"].eq(True)].copy()
eligible_signal_names = eligible_candidates["signal_name"].dropna().astype(str).unique().tolist()

candidate_tier_counts = (
    diversity_candidates["diversity_candidate_tier"]
    .value_counts(dropna=False)
    .rename_axis("diversity_candidate_tier")
    .reset_index(name="n_candidates")
)
orthogonal_diversifier_candidates = diversity_candidates.loc[
    diversity_candidates["diversity_candidate_tier"].eq("ORTHOGONAL_DIVERSIFIER")
].copy()
range_expansion_candidates = diversity_candidates.loc[
    diversity_candidates["signal_name"].eq("range_expansion_failure_5")
].copy()

print(f"Candidate rows from reproducibility gate: {len(diversity_candidates)}")
print(f"Eligible diversity candidates: {len(eligible_candidates)}")
print(f"Unique eligible signal names: {len(eligible_signal_names)}")
print("Candidate counts by diversity_candidate_tier")
display(candidate_tier_counts)
print("Orthogonal diversifier candidates admitted")
display(orthogonal_diversifier_candidates)
print("range_expansion_failure_5 candidate rows")
display(range_expansion_candidates)
display(diversity_candidates)

Candidate rows from reproducibility gate: 1
Eligible diversity candidates: 1
Unique eligible signal names: 1
Candidate counts by diversity_candidate_tier


,diversity_candidate_tier,n_candidates
0,ORTHOGONAL_DIVERSIFIER,1


Orthogonal diversifier candidates admitted


,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_version,signal_health_score,signal_health_gate,n_tests,n_passed,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate,run_id,reproducibility_version,signal_direction,signal_strength,recommended_use,regime_fragility_flag,decay_risk_flag,scoring_status,decay_status,diversity_candidate_tier,eligible_for_diversity
0,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,orthogonal_generated,phase2_orthogonal_signals_v2,volatility_structure,phase2_orthogonal_signals_v2,62.0,WATCHLIST_RESEARCH,14,12,0.857143,0.015966,0.00472,CONDITIONAL_PASS,WATCHLIST_ALPHA_RESEARCH,phase2_nb03f_signal_reproducibility_20260510_2...,phase2_signal_reproducibility_v1,POSITIVE_EDGE,WEAK,CONDITIONAL,HIGH_REGIME_FRAGILITY,LOW_DECAY_RISK,WATCHLIST,STABLE,ORTHOGONAL_DIVERSIFIER,True


range_expansion_failure_5 candidate rows


,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_version,signal_health_score,signal_health_gate,n_tests,n_passed,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate,run_id,reproducibility_version,signal_direction,signal_strength,recommended_use,regime_fragility_flag,decay_risk_flag,scoring_status,decay_status,diversity_candidate_tier,eligible_for_diversity


,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_version,signal_health_score,signal_health_gate,n_tests,n_passed,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate,run_id,reproducibility_version,signal_direction,signal_strength,recommended_use,regime_fragility_flag,decay_risk_flag,scoring_status,decay_status,diversity_candidate_tier,eligible_for_diversity
0,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,orthogonal_generated,phase2_orthogonal_signals_v2,volatility_structure,phase2_orthogonal_signals_v2,62.0,WATCHLIST_RESEARCH,14,12,0.857143,0.015966,0.00472,CONDITIONAL_PASS,WATCHLIST_ALPHA_RESEARCH,phase2_nb03f_signal_reproducibility_20260510_2...,phase2_signal_reproducibility_v1,POSITIVE_EDGE,WEAK,CONDITIONAL,HIGH_REGIME_FRAGILITY,LOW_DECAY_RISK,WATCHLIST,STABLE,ORTHOGONAL_DIVERSIFIER,True


## 5. Load candidate signal panels

In [5]:
candidate_signal_rows = load_candidate_signal_rows(eligible_signal_names, db_path=DB_PATH)
signal_panels = build_signal_panels(
    candidate_table=diversity_candidates,
    candidate_signals_long=candidate_signal_rows,
)

panel_summary = pd.DataFrame(
    [
        {
            "signal_key": signal_key,
            "n_dates": panel.shape[0],
            "n_tickers": panel.shape[1],
            "finite_pct": float(panel.notna().to_numpy().mean()) if panel.size else 0.0,
        }
        for signal_key, panel in signal_panels.items()
    ]
)

print(f"Loaded candidate signal rows: {len(candidate_signal_rows):,}")
print(f"Built signal panels: {len(signal_panels)}")
display(panel_summary)

Loaded candidate signal rows: 1,002,844
Built signal panels: 1


,signal_key,n_dates,n_tickers,finite_pct
0,vol_of_vol_20__h10,2098,478,0.880582


## 6. Build similarity matrix and diagnostics

In [6]:
signal_diversity_similarity = build_signal_similarity_matrix(
    candidate_table=diversity_candidates,
    signal_panels=signal_panels,
)
signal_diversity_diagnostics = compute_diversity_diagnostics(
    similarity=signal_diversity_similarity,
    candidate_table=diversity_candidates,
)

print("Diversity diagnostics")
display(signal_diversity_diagnostics)

display(signal_diversity_similarity.sort_values("correlation", key=lambda s: s.abs(), ascending=False).head(20))

Diversity diagnostics


,n_candidates,avg_abs_correlation,max_abs_correlation,median_abs_correlation,effective_signal_count
0,1,NaN,NaN,NaN,1.0


,signal_key_1,signal_key_2,signal_name_1,horizon_1,signal_name_2,horizon_2,correlation
0,vol_of_vol_20__h10,vol_of_vol_20__h10,vol_of_vol_20,10,vol_of_vol_20,10,1.0


## 7. Greedy diversity selection

In [7]:
signal_diversity_selection = greedy_diversity_selection(
    candidate_table=diversity_candidates,
    similarity=signal_diversity_similarity,
    correlation_threshold=CORRELATION_THRESHOLD,
    min_selected=MIN_SELECTED,
)

selected_signals = signal_diversity_selection.loc[signal_diversity_selection["selected_flag"].eq(1)].copy()
redundant_rejected_signals = signal_diversity_selection.loc[
    signal_diversity_selection["diversity_group"].eq("REDUNDANT_REJECTED")
].copy()
forced_selected_signals = signal_diversity_selection.loc[
    signal_diversity_selection["diversity_group"].eq("FORCED_MIN_SELECTED")
].copy()
orthogonal_selection_rows = signal_diversity_selection.loc[
    signal_diversity_selection["diversity_candidate_tier"].eq("ORTHOGONAL_DIVERSIFIER")
].copy()
range_expansion_selection_rows = signal_diversity_selection.loc[
    signal_diversity_selection["signal_name"].eq("range_expansion_failure_5")
].copy()

selection_group_counts = (
    signal_diversity_selection["diversity_group"]
    .value_counts(dropna=False)
    .rename_axis("diversity_group")
    .reset_index(name="n_signals")
)

selection_tier_counts = (
    signal_diversity_selection["diversity_candidate_tier"]
    .value_counts(dropna=False)
    .rename_axis("diversity_candidate_tier")
    .reset_index(name="n_signals")
)

if len(selected_signals) < MIN_SELECTED:
    print(f"Selected signals below MIN_SELECTED={MIN_SELECTED}; only {len(selected_signals)} eligible non-redundant candidates were available.")
elif not forced_selected_signals.empty:
    print("MIN_SELECTED was reached by force-selecting the next best core candidates after threshold relaxation.")

print(f"Selected signals: {len(selected_signals)}")
print("Selection group counts")
display(selection_group_counts)
print("Selection tier counts")
display(selection_tier_counts)
print("Selected signals including tier/source/cluster")
display(selected_signals)
print("Orthogonal diversifier selection rows")
display(orthogonal_selection_rows)
print("range_expansion_failure_5 selection rows")
display(range_expansion_selection_rows)
print("Rejected redundant signals")
display(redundant_rejected_signals)
display(signal_diversity_selection)

Selected signals below MIN_SELECTED=3; only 1 eligible non-redundant candidates were available.
Selected signals: 1
Selection group counts


,diversity_group,n_signals
0,ORTHOGONAL_DIVERSIFIER_SELECTED,1


Selection tier counts


,diversity_candidate_tier,n_signals
0,ORTHOGONAL_DIVERSIFIER,1


Selected signals including tier/source/cluster


...

range_expansion_failure_5 selection rows


,signal_key,signal_name,horizon,signal_family,diversity_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,repro_candidate_tier,signal_health_score,final_research_gate,reproducibility_status,pass_rate,avg_effective_mean_ic,selected_flag,selection_rank,max_corr_to_selected,selection_reason,diversity_group


Rejected redundant signals


,signal_key,signal_name,horizon,signal_family,diversity_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,repro_candidate_tier,signal_health_score,final_research_gate,reproducibility_status,pass_rate,avg_effective_mean_ic,selected_flag,selection_rank,max_corr_to_selected,selection_reason,diversity_group


,signal_key,signal_name,horizon,signal_family,diversity_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,repro_candidate_tier,signal_health_score,final_research_gate,reproducibility_status,pass_rate,avg_effective_mean_ic,selected_flag,selection_rank,max_corr_to_selected,selection_reason,diversity_group
0,vol_of_vol_20__h10,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_DIVERSIFIER,orthogonal_generated,phase2_orthogonal_signals_v2,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,62.0,WATCHLIST_ALPHA_RESEARCH,CONDITIONAL_PASS,0.857143,0.015966,1,1,0.0,Selected within correlation threshold 0.85.,ORTHOGONAL_DIVERSIFIER_SELECTED


## 8. Family diversity report

In [8]:
signal_diversity_family_report = build_family_diversity_report(
    candidate_table=diversity_candidates,
    selection=signal_diversity_selection,
    similarity=signal_diversity_similarity,
)

signal_diversity_cluster_report = build_cluster_diversity_report(
    candidate_table=diversity_candidates,
    selection=signal_diversity_selection,
    similarity=signal_diversity_similarity,
)

print("Family diversity report")
display(signal_diversity_family_report)
print("Cluster diversity report")
display(signal_diversity_cluster_report)

Family diversity report


,signal_family,n_candidates,n_selected,avg_health_score,max_health_score,avg_abs_corr_within_family
0,volatility_structure,1,1,62.0,62.0,NaN


Cluster diversity report


,orthogonal_cluster,n_candidates,n_selected,n_orthogonal_diversifiers,avg_health_score,max_health_score,avg_abs_corr_within_cluster
0,volatility_structure,1,1,1,62.0,62.0,NaN


## 9. Save outputs to SQLite

In [9]:
saved_paths = save_signal_diversity_outputs(
    similarity=signal_diversity_similarity,
    diagnostics=signal_diversity_diagnostics,
    selection=signal_diversity_selection,
    family_report=signal_diversity_family_report,
    cluster_report=signal_diversity_cluster_report,
    db_path=DB_PATH,
    run_id=run_id,
    diversity_version=DIVERSITY_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            "artifact": artifact,
            "current_table": tables[0],
            "history_table": tables[1],
            "sqlite_path": str(saved_paths[artifact]),
        }
        for artifact, tables in SIGNAL_DIVERSITY_TABLES.items()
    ]
)

display(sqlite_tables_written)

,artifact,current_table,history_table,sqlite_path
0,similarity,signal_diversity_similarity_current,signal_diversity_similarity_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,diagnostics,signal_diversity_diagnostics_current,signal_diversity_diagnostics_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,selection,signal_diversity_selection_current,signal_diversity_selection_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,family_report,signal_diversity_family_report_current,signal_diversity_family_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,cluster_report,signal_diversity_cluster_report_current,signal_diversity_cluster_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 10. Final summary

In [10]:
final_summary = pd.DataFrame(
    [
        {"metric": "run_id", "value": run_id},
        {"metric": "run_timestamp", "value": run_timestamp},
        {"metric": "diversity_version", "value": DIVERSITY_VERSION},
        {"metric": "candidate_count", "value": len(eligible_candidates)},
        {"metric": "selected_count", "value": int(signal_diversity_selection["selected_flag"].sum())},
        {"metric": "orthogonal_diversifier_candidates", "value": len(orthogonal_diversifier_candidates)},
        {"metric": "orthogonal_diversifier_selected", "value": int(selected_signals["diversity_candidate_tier"].eq("ORTHOGONAL_DIVERSIFIER").sum())},
        {"metric": "redundant_rejected_count", "value": len(redundant_rejected_signals)},
    ]
)

print("Candidate count")
display(final_summary)

print("Candidate counts by diversity_candidate_tier")
display(candidate_tier_counts)

print("Orthogonal diversifier candidates admitted")
display(orthogonal_diversifier_candidates)

print("Diversity diagnostics")
display(signal_diversity_diagnostics)

print("Selected signals")
display(selected_signals)

print("Rows for range_expansion_failure_5")
display(range_expansion_selection_rows)

print("Redundant rejected signals")
display(redundant_rejected_signals)

print("Family diversity report")
display(signal_diversity_family_report)

print("Cluster diversity report")
display(signal_diversity_cluster_report)

print("SQLite tables written")
display(sqlite_tables_written)

Candidate count


,metric,value
0,run_id,phase2_signal_diversity_20260510_225559
1,run_timestamp,2026-05-10 22:55:59
2,diversity_version,phase2_signal_diversity_v1
3,candidate_count,1
4,selected_count,1
5,orthogonal_diversifier_candidates,1
6,orthogonal_diversifier_selected,1
7,redundant_rejected_count,0


Candidate counts by diversity_candidate_tier


,diversity_candidate_tier,n_candidates
0,ORTHOGONAL_DIVERSIFIER,1


Orthogonal diversifier candidates admitted


...

,signal_family,n_candidates,n_selected,avg_health_score,max_health_score,avg_abs_corr_within_family
0,volatility_structure,1,1,62.0,62.0,NaN


Cluster diversity report


,orthogonal_cluster,n_candidates,n_selected,n_orthogonal_diversifiers,avg_health_score,max_health_score,avg_abs_corr_within_cluster
0,volatility_structure,1,1,1,62.0,62.0,NaN


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,similarity,signal_diversity_similarity_current,signal_diversity_similarity_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,diagnostics,signal_diversity_diagnostics_current,signal_diversity_diagnostics_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,selection,signal_diversity_selection_current,signal_diversity_selection_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,family_report,signal_diversity_family_report_current,signal_diversity_family_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,cluster_report,signal_diversity_cluster_report_current,signal_diversity_cluster_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


In [11]:
from src.db import load_table

sel = load_table("signal_diversity_selection_current")
diag = load_table("signal_diversity_diagnostics_current")
fam = load_table("signal_diversity_family_report_current")
cluster = load_table("signal_diversity_cluster_report_current")

display(diag)
display(sel.sort_values(["selected_flag", "selection_rank"], ascending=[False, True]))
display(fam)
display(cluster)

,n_candidates,avg_abs_correlation,max_abs_correlation,median_abs_correlation,effective_signal_count,run_id,diversity_version
0,1,None,None,None,1.0,phase2_signal_diversity_20260510_225559,phase2_signal_diversity_v1


,signal_key,signal_name,horizon,signal_family,diversity_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,repro_candidate_tier,signal_health_score,final_research_gate,reproducibility_status,pass_rate,avg_effective_mean_ic,selected_flag,selection_rank,max_corr_to_selected,selection_reason,diversity_group,run_id,diversity_version
0,vol_of_vol_20__h10,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_DIVERSIFIER,orthogonal_generated,phase2_orthogonal_signals_v2,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,62.0,WATCHLIST_ALPHA_RESEARCH,CONDITIONAL_PASS,0.857143,0.015966,1,1,0.0,Selected within correlation threshold 0.85.,ORTHOGONAL_DIVERSIFIER_SELECTED,phase2_signal_diversity_20260510_225559,phase2_signal_diversity_v1


,signal_family,n_candidates,n_selected,avg_health_score,max_health_score,avg_abs_corr_within_family,run_id,diversity_version
0,volatility_structure,1,1,62.0,62.0,None,phase2_signal_diversity_20260510_225559,phase2_signal_diversity_v1


,orthogonal_cluster,n_candidates,n_selected,n_orthogonal_diversifiers,avg_health_score,max_health_score,avg_abs_corr_within_cluster,run_id,diversity_version
0,volatility_structure,1,1,1,62.0,62.0,None,phase2_signal_diversity_20260510_225559,phase2_signal_diversity_v1


In [12]:
from src.db import load_table

meta = load_table("candidate_signal_metadata_current")
quality = load_table("candidate_signal_quality_current")
signals = load_table("candidate_signals_current")
scores = load_table("signal_scores_current")
health = load_table("signal_health_score_current")
div = load_table("signal_diversity_selection_current")

new = ["volume_acceleration_20", "price_volume_divergence_20"]

print("Metadata rows:")
display(meta[meta["signal_name"].isin(new)])

print("Quality rows:")
display(quality[quality["signal_name"].isin(new)])

print("Signal row counts / non-null:")
display(
    signals[signals["signal_name"].isin(new)]
    .groupby("signal_name")["signal_value"]
    .agg(["count", "size", "mean", "std", "min", "max"])
)

print("Scoring rows:")
display(scores[scores["signal_name"].isin(new)].sort_values(["signal_name", "horizon"]))

print("Health rows:")
display(health[health["signal_name"].isin(new)].sort_values(["signal_name", "horizon"]))

print("Diversity rows:")
display(div[div["signal_name"].isin(new)])

Metadata rows:


,signal_name,signal_family,formula_type,parameters,data_dependencies,lookback,direction_convention,input_fields,normalization_notes,normalization,signal_source,discovery_family,discovery_version,signal_template_name,parameter_config_json,signal_version,run_id,timestamp,created_timestamp,notes,orthogonal_version
22,volume_acceleration_20,volume_flow,rolling_volume_mean_change,lookback=20,volume,20,higher_is_accelerating_volume,volume,Raw trailing signal is cross-sectionally z-sco...,cross_sectional_zscore_by_date,manual_core,,,,,phase2_candidate_v1,phase2_nb02_20260507_224108,2026-05-07 22:41:08,2026-05-07,Current 20-day average volume divided by the p...,None
23,price_volume_divergence_20,volume_flow,price_return_minus_volume_acceleration,lookback=20,"close,volume",20,higher_is_price_strength_less_supported_by_vol...,"close,volume",Raw trailing signal is cross-sectionally z-sco...,cross_sectional_zscore_by_date,manual_core,,,,,phase2_candidate_v1,phase2_nb02_20260507_224108,2026-05-07 22:41:08,2026-05-07,20-day price return minus 20-day volume accele...,None


Quality rows:


,signal_name,signal_family,n_dates,n_tickers,missing_pct,finite_pct,first_valid_date,last_valid_date,run_id,signal_version,signal_source,orthogonal_version,orthogonal_cluster,status
22,volume_acceleration_20,volume_flow,2098,478,0.472327,0.527673,2018-03-21,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1,None,None,None,None
23,price_volume_divergence_20,volume_flow,2098,478,0.472327,0.527673,2018-03-21,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1,None,None,None,None


Signal row counts / non-null:


...

,signal_name,horizon,method,n_obs,mean_ic,median_ic,ic_std,ic_ir,hit_rate,positive_ic_rate,missing_pct,signal_family,signal_version,run_id,scoring_version


Health rows:


,signal_name,horizon,signal_family,signal_direction,signal_strength,best_mean_ic,best_abs_mean_ic,best_ic_ir,scoring_status,decay_status,decay_risk_flag,mean_rolling_ic,recent_ic,early_ic,ic_change,sign_stability,adjusted_best_abs_ic,recommended_use,regime_fragility_flag,regime_consistency_score,regime_sample_weight,wfv_status,direction_flip_warning,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,signal_health_score,signal_health_gate,health_notes,run_id,health_version


Diversity rows:


,signal_key,signal_name,horizon,signal_family,diversity_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,repro_candidate_tier,signal_health_score,final_research_gate,reproducibility_status,pass_rate,avg_effective_mean_ic,selected_flag,selection_rank,max_corr_to_selected,selection_reason,diversity_group,run_id,diversity_version


In [13]:
sim = load_table("signal_diversity_similarity_current")

display(
    sim[
        sim["signal_name_1"].isin(new) | sim["signal_name_2"].isin(new)
    ].sort_values("correlation", key=lambda s: s.abs(), ascending=False)
)

,signal_key_1,signal_key_2,signal_name_1,horizon_1,signal_name_2,horizon_2,correlation,run_id,diversity_version
